In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

Tabela dimensão mestre. Une as 4 tabelas de cadastro e resolve o join uma única vez. 

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")

    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. Fundos Diarios

In [0]:
silver_table_cvm_day = "workspace.case_spark_cvm.silver_cvm_fundos_diario"

df_cvm_fundos_diario_silver = ler_ultima_particao_tabela_spark(spark, silver_table_cvm_day)

In [0]:
display(df_cvm_fundos_diario_silver)

In [0]:
df_cvm_fundos_diario_silver.orderBy(f.col("dt_comptc").desc()).show()

### 2. Registros Classes

In [0]:
silver_table_classe_cvm = "workspace.case_spark_cvm.silver_registro_classe_cvm"

df_registro_classe_cvm_silver = ler_ultima_particao_tabela_spark(spark, silver_table_classe_cvm)

In [0]:
display(df_registro_classe_cvm_silver)

### 3. Registros Fundos

In [0]:
silver_table_fundos_cvm = "workspace.case_spark_cvm.silver_registro_fundo_cvm"

df_registro_fundo_cvm_silver = ler_ultima_particao_tabela_spark(spark, silver_table_fundos_cvm)

In [0]:
display(df_registro_fundo_cvm_silver)

### 4. Registros Subclasses

In [0]:
silver_table_subclasse_cvm = "workspace.case_spark_cvm.silver_registro_fundo_cvm"

df_registro_subclasse_cvm_silver = ler_ultima_particao_tabela_spark(spark, silver_table_subclasse_cvm)

In [0]:
display(df_registro_subclasse_cvm_silver)

In [0]:
df_cvm_fundos_diario_silver.groupBy('id_subclasse').count().show()

### 5. Joins

In [0]:
display(df_cvm_fundos_diario_silver)

In [0]:
d = df_cvm_fundos_diario_silver.alias("d")
f = df_registro_fundo_cvm_silver.alias("f")
c = df_registro_classe_cvm_silver.alias("c")
sc = df_registro_subclasse_cvm_silver.alias("sc")

df_dim_fundo = d\
 .join(
    f,
    d.cnpj_fundo_classe == f.cnpj_fundo,
    "left"
).join(
    c,
    f.id_registro_fundo == c.id_registro_fundo,
    "left"
).join(
    sc,
    f.id_registro_fundo == sc.id_registro_fundo,
    "left"
)


In [0]:
df_dim_fundo = df_dim_fundo.select(
    d.cnpj_fundo_classe,
    d.id_subclasse,
    sc.denominacao_social,
    f.tipo_fundo,
    c.tipo_classe,
    c.classificacao,
    c.classificacao_anbima,
    c.situacao,
    c.forma_condominio,
    c.publico_alvo,
    c.classe_esg,
    c.exclusivo,
    f.gestor,
    f.administrador,
    c.custodiante,
    c.data_inicio,
    c.indicador_desempenho
).dropDuplicates(["cnpj_fundo_classe"])

In [0]:
df_dim_fundo = df_dim_fundo.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
display(df_dim_fundo)

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_dim_fundo.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.gold_dim_fundo")